In [1]:
import os
import torch
import safetensors
import json

In [ ]:
model_path = "/mnt/hdd/data/yxlin/huggingface/Meta-Llama-2-7B"
safetensor_files = [name for name in os.listdir(model_path) if name.endswith(".safetensors")]

# 假设我需要读取如下layer的weight, 应该如何做？
model_layer_key = "model.layers.9.post_attention_layernorm.weight"

# 我先要读取 model_path 下一个名为 model.safetensors.index.json 的文件，其记录着model layer key 和 safetensors的关系
safetensor_index_path = os.path.join(model_path, "model.safetensors.index.json")
f = open(safetensor_index_path, "r", encoding="utf-8")
safetensor_index = json.load(f)["weight_map"]

# safetensor_index 是个字典
safetensor_file_name = safetensor_index[model_layer_key]
safetensor_file_path = os.path.join(model_path, safetensor_file_name)

# 我们需要使用 safetensors 打开 safetensor 文件
with safetensors.safe_open(safetensor_file_path, framework="pt", device="cuda") as f:
    print(type(f))
    # f 是个类，有相关方法依据 model layer key 获得tensor
    tensor = f.get_tensor(model_layer_key)

print(tensor.shape)

<class 'builtins.safe_open'>
torch.Size([4096])


所以可以总结下在 weight.py 下，作者是如何从 .safetensors 文件中加载权重到model中的

首先作者 将 model architecture 和 model weight 分离， 专门用一个名为 LlamaWeight Class 装载 model weight

LlamaWeight Class 依旧依据模型架构是一种 递归式 的类。例如 LlamaWeight Class 包含 LlamaTransformerLayerWeight Class

LlamaTransformerLayerWeight Class 和 LlamaWeight Class 基类均为 WeightBase Class

上述 Class 会依赖一个列表registered_weights, 列表中元素为 RegisteredWeightItem Class，其属性是model layer name and key等

我们会实现一个方法getter,其可以通过 RegisteredWeightItem Class 中定义的model layer name key 从safetensor文件中获取出 model layer对应的weight

所以 WeightBase Class load_weight 函数做的事情就是 遍历registered_weights列表，对列表中的每一个RegisteredWeightItem Class调用getter函数

我们可以通过这种方式递归地实现将model layer weight全部加载